# Lecture 2: Python for Text Processing
### NLP Course 2027

---

## Learning Outcomes
- Confidently manipulate text using Python string methods
- Apply tokenization and sentence segmentation with NLTK
- Normalize text through lowercasing, punctuation removal, and stemming
- Read and process raw text files

**Primary Reference:** *NLP with Python* (Bird, Klein, Loper), Chapter 3

## 1. Python String Operations for NLP

Python's built-in string methods are the foundation of all text processing.

In [11]:
text = 'Natural Language Processing is Fascinating!'

# Case operations
print('lower:', text.lower())
print('upper:', text.upper())
print('title:', text.title())

# Inspection
print(f'Length: {len(text)}')
print(f'Starts with Natural: {text.startswith("Natural")}')
print(f'Contains Lang: {"Lang" in text}')
print(f'Count of "a": {text.count("a")}')

lower: natural language processing is fascinating!
upper: NATURAL LANGUAGE PROCESSING IS FASCINATING!
title: Natural Language Processing Is Fascinating!
Length: 43
Starts with Natural: True
Contains Lang: True
Count of "a": 6


In [12]:
# Splitting and joining
sentence = 'the quick brown fox jumps over the lazy dog'
words = sentence.split()
print('split():', words)

csv = 'apple,banana,cherry'
print('split by comma:', csv.split(','))

tokens = ['Natural', 'Language', 'Processing']
print('join with space:', ' '.join(tokens))
print('join with hyphen:', '-'.join(tokens))

split(): ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
split by comma: ['apple', 'banana', 'cherry']
join with space: Natural Language Processing
join with hyphen: Natural-Language-Processing


In [13]:
# List comprehensions for text processing
words = 'The quick brown fox jumps over the lazy dog'.split()

# Filter: words longer than 3 chars
long_words = [w for w in words if len(w) > 3]
print('Long words:', long_words)

# Transform: lowercase all
lower_words = [w.lower() for w in words]
print('Lowercase:', lower_words)

# Filter + transform
t_words = [w.lower() for w in words if w.lower().startswith('t')]
print('Words starting with t:', t_words)

# Keep only alphabetic
noisy = ['hello', '123', 'world!', 'foo', '42']
alpha = [w for w in noisy if w.isalpha()]
print('Alpha only:', alpha)

Long words: ['quick', 'brown', 'jumps', 'over', 'lazy']
Lowercase: ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
Words starting with t: ['the', 'the']
Alpha only: ['hello', 'foo']


## 2. Tokenization

**Tokenization** splits text into meaningful units (tokens).

```
Input:  'Dr. Smith visited the U.S.A. last week.'
               ↓  Tokenizer
Output: ['Dr.', 'Smith', 'visited', 'the', 'U.S.A.', 'last', 'week', '.']
```

Simple `.split()` fails on contractions, punctuation, abbreviations. NLTK tokenizers handle these.

**Key tokenizers:**
- `word_tokenize()` — general-purpose, handles punctuation
- `sent_tokenize()` — sentence boundaries
- `TweetTokenizer` — social media text
- `RegexpTokenizer` — custom patterns

In [14]:
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

tricky = "Dr. Smith visited the U.S.A. He couldn't believe it cost $1,200.50!"

print('simple split():    ', tricky.split()[:8])
print('word_tokenize():', word_tokenize(tricky))

simple split():     ['Dr.', 'Smith', 'visited', 'the', 'U.S.A.', 'He', "couldn't", 'believe']
word_tokenize(): ['Dr.', 'Smith', 'visited', 'the', 'U.S.A', '.', 'He', 'could', "n't", 'believe', 'it', 'cost', '$', '1,200.50', '!']


In [15]:
# Sentence tokenization: handles abbreviations correctly
paragraph = """Mr. Johnson went to Washington D.C. He met with Sen. Harris.
The meeting lasted 2 hrs. They discussed U.S. foreign policy."""

sents = sent_tokenize(paragraph)
print(f'{len(sents)} sentences detected:')
for i, s in enumerate(sents, 1):
    print(f'  [{i}] {s}')

4 sentences detected:
  [1] Mr. Johnson went to Washington D.C.
  [2] He met with Sen. Harris.
  [3] The meeting lasted 2 hrs.
  [4] They discussed U.S. foreign policy.


In [16]:
from nltk.tokenize import TweetTokenizer, RegexpTokenizer

tweet = '@elonmusk I cant believe #tesla is so AMAZING!!! 😍 http://t.co/xyz'

tt = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True)
print('TweetTokenizer:', tt.tokenize(tweet))

# Custom: keep only alpha tokens
alpha_tok = RegexpTokenizer(r'[a-zA-Z]+')
print('Alpha tokenizer:', alpha_tok.tokenize(tweet))

TweetTokenizer: ['i', 'cant', 'believe', '#tesla', 'is', 'so', 'amazing', '!', '!', '!', '😍', 'http://t.co/xyz']
Alpha tokenizer: ['elonmusk', 'I', 'cant', 'believe', 'tesla', 'is', 'so', 'AMAZING', 'http', 't', 'co', 'xyz']


## 3. Text Normalization

**Normalization** reduces text variation to a canonical form.

```
Step 1: Lowercase      → 'Hello World' → 'hello world'
Step 2: Remove punct   → 'hello!' → 'hello'
Step 3: Remove stops   → 'the hello' → 'hello'
Step 4: Stem/Lemmatize → 'running' → 'run'
```

### Stemming vs Lemmatization
| | Stemming | Lemmatization |
|-|----------|---------------|
| Speed | Fast | Slower |
| Method | Rule-based | Dictionary-based |
| Output | May be non-word ("studi") | Always valid word |
| Use case | Search, IR | Text understanding |

In [17]:
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

def normalize_text(text, stem=False, lemmatize=True):
    """Complete normalization pipeline."""
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in string.punctuation]
    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [t for t in tokens if t.isalpha()]
    if stem:
        ps = PorterStemmer()
        tokens = [ps.stem(t) for t in tokens]
    elif lemmatize:
        lem = WordNetLemmatizer()
        tokens = [lem.lemmatize(t) for t in tokens]
    return tokens

sample = 'The researchers are studying 3 NLP models and evaluating performance!'
print('Original:  ', sample)
print('Normalized:', normalize_text(sample))

Original:   The researchers are studying 3 NLP models and evaluating performance!
Normalized: ['researcher', 'studying', 'nlp', 'model', 'evaluating', 'performance']


In [18]:
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

ps = PorterStemmer()
lem = WordNetLemmatizer()

words = ['running', 'studies', 'wolves', 'better', 'corpora', 'caring']

print(f'{"Word":<15} {"Stemmed":<15} {"Lemmatized"}')
print('-' * 45)
for w in words:
    print(f'{w:<15} {ps.stem(w):<15} {lem.lemmatize(w)}')

Word            Stemmed         Lemmatized
---------------------------------------------
running         run             running
studies         studi           study
wolves          wolv            wolf
better          better          better
corpora         corpora         corpus
caring          care            caring


## 4. Working with Text Files

In practice you read from files, web scrapes, or APIs. Key considerations:
- **Encoding**: Always use `encoding='utf-8'`
- **Cleaning**: Strip HTML, headers, footers
- **Memory**: For large files, iterate line by line

## Practice Exercises

See **`Lecture-02-Homework.ipynb`** for the practice exercises accompanying this lecture.

In [19]:
# Write a sample file then process it
content = """Chapter 1: Introduction
Natural Language Processing (NLP) is a field of artificial intelligence.
It focuses on enabling computers to understand human language.

Chapter 2: Applications
NLP powers search engines, chatbots, and machine translators.
Companies like Google and Amazon invest heavily in NLP research."""

with open('/tmp/nlp_sample.txt', 'w', encoding='utf-8') as f:
    f.write(content)

with open('/tmp/nlp_sample.txt', 'r', encoding='utf-8') as f:
    raw = f.read()

print(f'Read {len(raw)} chars')
print(raw)

Read 311 chars
Chapter 1: Introduction
Natural Language Processing (NLP) is a field of artificial intelligence.
It focuses on enabling computers to understand human language.

Chapter 2: Applications
NLP powers search engines, chatbots, and machine translators.
Companies like Google and Amazon invest heavily in NLP research.


In [20]:
from nltk import FreqDist

# Full pipeline on file content
tokens = normalize_text(raw)
fdist = FreqDist(tokens)

print('Top 10 content words:')
for word, cnt in fdist.most_common(10):
    print(f'  {word}: {cnt}')

Top 10 content words:
  nlp: 3
  chapter: 2
  language: 2
  introduction: 1
  natural: 1
  processing: 1
  field: 1
  artificial: 1
  intelligence: 1
  focus: 1


## Summary

| Tool | Purpose |
|------|---------|
| `str.lower()`, `split()` | Basic string manipulation |
| `word_tokenize()` | Smarter word splitting |
| `sent_tokenize()` | Sentence boundary detection |
| `stopwords` | Remove high-frequency function words |
| `PorterStemmer` | Fast rule-based normalization |
| `WordNetLemmatizer` | Accurate dictionary-based normalization |

**Next Lecture**: Corpus Access & Lexical Resources — NLTK corpora and WordNet.

---
*Book references: NLP with Python Ch.3 | Practical NLP Ch.2*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**